# Preprocessing and Feature Transformation

In [2]:
import pandas as pd
import numpy as np

np.random.seed(42)

n = 1000

df = pd.DataFrame({
    "customer_id": range(1001, 1001 + n),
    "age": np.random.randint(18, 71, n).astype(float),
    "income": np.random.normal(55000, 18000, n),
    "account_tenure_months": np.random.randint(1, 121, n),
    "monthly_spend": np.random.gamma(4, 800, n),
    "support_tickets": np.random.poisson(2, n).astype(float),
    "membership": np.random.choice(
        ["Basic", "Silver", "Gold"], n
    ),
    "city": np.random.choice(
        ["Delhi", "Mumbai", "Pune", "Jaipur"], n
    ),
    "marketing_channel": np.random.choice(
        ["Email", "Social", "Paid Search", "Organic"], n
    ),
    "churn": np.random.choice([0, 1], n, p=[0.75, 0.25])
})

# Introduce missing values
for col in ["age", "income", "monthly_spend", "support_tickets"]:
    idx = np.random.choice(df.index, 40, replace=False)
    df.loc[idx, col] = np.nan

for col in ["membership", "city", "marketing_channel"]:
    idx = np.random.choice(df.index, 30, replace=False)
    df.loc[idx, col] = np.nan

df.head()

,customer_id,age,income,account_tenure_months,monthly_spend,support_tickets,membership,city,marketing_channel,churn
0,1001,56.0,69394.442007,108,2554.479852,1.0,Basic,Pune,Social,0
1,1002,69.0,56242.099204,5,6127.006202,3.0,Basic,Delhi,Social,0
2,1003,46.0,68837.094859,3,4135.756775,2.0,Basic,Mumbai,Organic,0
3,1004,32.0,40953.819627,65,2451.101322,3.0,NaN,Delhi,Organic,0
4,1005,60.0,87053.129887,4,2772.010180,NaN,Basic,Delhi,Paid Search,0


#### Q1. Identify preprocessing requirements
Identify numerical and categorical columns in df.
- Count missing values per column.
- Exclude customer_id and churn from the predictor features.
- Create X and y.
- Create numeric_features and categorical_features dynamically using dtypes.

In [4]:
num_cols=df.select_dtypes(include=['int','float']).columns.tolist()
cat_cols=df.select_dtypes(include='object').columns.tolist()
display('numeric columns', num_cols,'categorical columns',cat_cols)

'numeric columns'

['customer_id',
 'age',
 'income',
 'account_tenure_months',
 'monthly_spend',
 'support_tickets',
 'churn']

'categorical columns'

['membership', 'city', 'marketing_channel']

In [5]:
df.isnull().sum()

customer_id               0
age                      40
income                   40
account_tenure_months     0
monthly_spend            40
support_tickets          40
membership               30
city                     30
marketing_channel        30
churn                     0
dtype: int64

In [6]:
y=df['churn']
X=df.drop(columns=['churn','customer_id'])
y.shape,X.shape

((1000,), (1000, 8))

In [7]:
df.shape

(1000, 10)

#### Q2. Split before preprocessing
Use train_test_split() to create an 80/20 split with random_state=42 and stratify=y.

Verify the shapes of the training and testing datasets.

Important: Do not impute or scale the full dataset before splitting.

In [9]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
print(X_train.shape,X_test.shape)

(800, 8) (200, 8)


#### Q3. Impute numerical features
Use SimpleImputer(strategy="median") to handle missing numerical values.
- Fit the imputer on X_train.
- Transform both X_train and X_test.
- Verify that missing numerical values have been handled.
- Explain why fitting on the full dataset would cause leakage.

In [11]:
from sklearn.impute import SimpleImputer
imputer=SimpleImputer(strategy='median')

In [12]:
num_cols=X.select_dtypes(include=['int','float']).columns.tolist()
X_train[num_cols]=imputer.fit_transform(X_train[num_cols])
X_test[num_cols]=imputer.transform(X_test[num_cols])
X_train[num_cols].isnull().sum()

age                      0
income                   0
account_tenure_months    0
monthly_spend            0
support_tickets          0
dtype: int64

#### Q4. Impute categorical features
Use SimpleImputer(strategy="most_frequent") on the categorical features.

Verify that both training and testing datasets contain no missing categorical values after transformation.

In [14]:
cat_cols=X.select_dtypes(include='object').columns.tolist()
imputer=SimpleImputer(strategy='most_frequent')
X_train[cat_cols]=imputer.fit_transform(X_train[cat_cols])
X_test[cat_cols]=imputer.transform(X_test[cat_cols])
X_train[cat_cols].isnull().sum()

membership           0
city                 0
marketing_channel    0
dtype: int64

#### Q5. Apply one-hot encoding
Use OneHotEncoder(handle_unknown="ignore") to encode the categorical columns.
- Fit on training data only.
- Transform the training and testing datasets.
- Display the encoded feature names.
- Explain what happens when the test set contains a category not seen during training.

In [16]:
from sklearn.preprocessing import OneHotEncoder
encoder=OneHotEncoder(handle_unknown='ignore',sparse_output=False)
X_train_encoded=pd.DataFrame(encoder.fit_transform(X_train[cat_cols]),
                            columns=encoder.get_feature_names_out(cat_cols),
                            index=X_train.index)
X_test_encoded=pd.DataFrame(encoder.transform(X_test[cat_cols]),
                            columns=encoder.get_feature_names_out(cat_cols),
                            index=X_test.index)
X_train_encoded

,membership_Basic,membership_Gold,membership_Silver,city_Delhi,city_Jaipur,city_Mumbai,city_Pune,marketing_channel_Email,marketing_channel_Organic,marketing_channel_Paid Search,marketing_channel_Social
5,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
78,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
312,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
635,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
437,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
579,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
694,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
973,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
552,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


#### Q6. Apply standard scaling
Use StandardScaler on the numerical features.

Compare the means and standard deviations before and after scaling.

Explain why the scaler must be fitted only on the training data.

In [18]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()

In [19]:
X_train_scaled=pd.DataFrame(
    scaler.fit_transform(X_train[num_cols]),
    columns=num_cols,
    index=X_train.index)

X_test_scaled=pd.DataFrame(
    scaler.transform(X_test[num_cols]),
    columns=num_cols,
    index=X_test.index)

display(X_train_scaled,'mean before scaling',X_train[num_cols].mean(),'mean after scaling',X_train_scaled[num_cols].mean())

,age,income,account_tenure_months,monthly_spend,support_tickets
5,-1.283339,0.205254,1.314865,0.269392,0.851243
78,-0.090460,-0.458560,-1.618108,-0.561840,-0.712464
312,-0.753171,0.062920,-1.066361,-0.142068,0.069389
635,-0.951984,0.079267,1.169668,0.028547,0.851243
437,-1.349610,-0.261703,-1.356754,0.458436,0.069389
...,...,...,...,...,...
579,-1.747237,-2.256577,-1.298676,-0.372808,0.851243
694,-0.686900,-0.294417,-1.269636,-1.147710,0.069389
973,0.903606,-1.620920,1.256786,-1.561614,-1.494317
552,-1.150797,0.424763,1.169668,-0.239033,1.633096


'mean before scaling'

age                         44.365000
income                   55676.978157
account_tenure_months       58.721250
monthly_spend             3244.915918
support_tickets              1.911250
dtype: float64

'mean after scaling'

age                     -1.332268e-16
income                  -1.110223e-17
account_tenure_months    5.884182e-17
monthly_spend            3.530509e-16
support_tickets          8.659740e-17
dtype: float64

#### Q7. Compare scaling methods
Apply both StandardScaler and MinMaxScaler to the numerical features.

Compare the resulting distributions and ranges. Explain when each method might be useful and why neither automatically eliminates the influence of outliers.

In [21]:
from sklearn.preprocessing import MinMaxScaler
scaler=MinMaxScaler()
X_train_minmax=pd.DataFrame(
    scaler.fit_transform(X_train[num_cols]),
    columns=num_cols,
    index=X_train.index)

X_test_minmax=pd.DataFrame(
    scaler.transform(X_test[num_cols]),
    columns=num_cols,
    index=X_test.index)
X_train_minmax

,age,income,account_tenure_months,monthly_spend,support_tickets
5,0.134615,0.566645,0.865546,0.341070,0.500000
78,0.480769,0.463582,0.016807,0.206703,0.166667
312,0.288462,0.544546,0.176471,0.274558,0.333333
635,0.230769,0.547084,0.823529,0.302138,0.500000
437,0.115385,0.494146,0.092437,0.371629,0.333333
...,...,...,...,...,...
579,0.000000,0.184427,0.109244,0.237259,0.500000
694,0.307692,0.489067,0.117647,0.111998,0.333333
973,0.769231,0.283117,0.848739,0.045091,0.000000
552,0.173077,0.600725,0.823529,0.258884,0.666667


In [22]:
print("Mean after StandardScaler:")
display(X_train_scaled[num_cols].mean())

print("Mean after MinMaxScaler:")
display(X_train_minmax[num_cols].mean())

Mean after StandardScaler:


age                     -1.332268e-16
income                  -1.110223e-17
account_tenure_months    5.884182e-17
monthly_spend            3.530509e-16
support_tickets          8.659740e-17
dtype: float64

Mean after MinMaxScaler:


age                      0.507019
income                   0.534777
account_tenure_months    0.485053
monthly_spend            0.297523
support_tickets          0.318542
dtype: float64

- StandardScaler: Useful for Logistic Regression, SVM, PCA, and KNN, where differences in feature scales can affect model training or distance calculations.

- MinMaxScaler: Useful for neural networks and other workflows that benefit from features within a fixed range, such as [0, 1].

#### Q8. Build a ColumnTransformer
Create a preprocessing transformer that applies:
- Median imputation followed by standard scaling to numerical features.
- Most-frequent imputation followed by one-hot encoding to categorical features.
Use Pipeline for each feature group and combine them with ColumnTransformer.

Fit it on X_train and transform both splits.

In [46]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [48]:
# 1. Numerical preprocessing pipeline
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 2. Categorical preprocessing pipeline
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# 3. Combine both pipelines
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, num_cols),
    ('cat', categorical_pipeline, cat_cols)
])

# 4. Fit on training data and transform both splits
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Training shape:", X_train_processed.shape)
print("Testing shape:", X_test_processed.shape)

print("Same number of features:",
      X_train_processed.shape[1] == X_test_processed.shape[1])

Training shape: (800, 16)
Testing shape: (200, 16)
Same number of features: True


#### Q9. Inspect the transformed matrix
Using your fitted ColumnTransformer:
- Retrieve the transformed feature names.
- Check the dimensions of the transformed training and testing matrices.
- Verify that no missing values remain.
- Explain why the transformed feature count may exceed the original number of columns.

In [59]:
feature_names = preprocessor.get_feature_names_out()
feature_names

array(['num__age', 'num__income', 'num__account_tenure_months',
       'num__monthly_spend', 'num__support_tickets',
       'cat__membership_Basic', 'cat__membership_Gold',
       'cat__membership_Silver', 'cat__city_Delhi', 'cat__city_Jaipur',
       'cat__city_Mumbai', 'cat__city_Pune',
       'cat__marketing_channel_Email', 'cat__marketing_channel_Organic',
       'cat__marketing_channel_Paid Search',
       'cat__marketing_channel_Social'], dtype=object)

In [61]:
print("Training matrix:", X_train_processed.shape)
print("Testing matrix:", X_test_processed.shape)

print("Number of transformed features:", len(feature_names))

Training matrix: (800, 16)
Testing matrix: (200, 16)
Number of transformed features: 16


In [63]:
print("Missing values in training:", np.isnan(X_train_processed).sum())
print("Missing values in testing:", np.isnan(X_test_processed).sum())

Missing values in training: 0
Missing values in testing: 0


#### Q10. Create a reusable preprocessing workflow
Write a function named build_preprocessor() that accepts numerical and categorical column lists and returns a fitted-free ColumnTransformer containing the required pipelines.

Test it by creating a new preprocessor, fitting it on X_train, and transforming both datasets.